# mini-vLLM — Kaggle T4 benchmark run

These are the README numbers. Settings: **GPU T4 x1**, **Internet on**.

1. Push your commits to GitHub first (Kaggle clones the public repo).
2. Run all cells, about 30–45 min of GPU time.
3. Download `results.csv` from the output panel and copy it into your local repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!git clone -q https://github.com/Taral92/mini-vllm.git
%cd mini-vllm
!git log --oneline -1
!pip install -q -e ".[dev]"
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## 1. Correctness gates (must all pass before any number counts)

In [ ]:
!pytest -q

## 2. Length sweep, batch 1: naive vs KV cache

In [ ]:
!mini-vllm-benchmark --device cuda --stages naive,kv --output-lens 32,64,128,256,512 --iters 3

## 3. Batch-size sweep: weights read once, reused by the whole batch

In [ ]:
for bs in (1, 4, 8, 16, 32):
    !mini-vllm-benchmark --device cuda --stages static,continuous --batch-size {bs} --output-lens 256 --iters 3

## 4. Mixed output lengths: where continuous batching beats static

In [ ]:
!mini-vllm-benchmark --device cuda --stages static,continuous --workload mixed --batch-size 8 --output-lens 256 --iters 3

## 5. Results

In [ ]:
import pandas as pd
df = pd.read_csv("results.csv")
cols = ["stage", "batch_size", "output_len", "tokens_per_sec", "ttft_ms", "tpot_ms", "p50_ms", "p99_ms", "peak_mem_gb"]
print(df[df.device == "cuda"][cols].round(2).to_markdown(index=False))

In [ ]:
import shutil
shutil.copy("results.csv", "/kaggle/working/results.csv")
print("download results.csv from the Output panel")